<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Módulo 2: Identificación y taxonomía de datos</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>

> **Idea central:** antes de transformar una variable, identifica qué representa, qué operaciones tienen sentido y qué información puede perderse al codificarla.

**En este archivo de jupyter vamos a tratar los siguientes puntos:**

1. Distinguir variables cualitativas, cuantitativas, binarias, nominales y ordinales.
2. Reconocer la diferencia entre el tipo almacenado (`dtype`) y la escala de medición.
3. Diagnosticar variables en un conjunto de datos real con `pandas`.
4. Identificar valores faltantes y elegir una estrategia con criterio.

Al final, usa la sección **Actividad de cierre** para comprobar tus decisiones.


## ¿Qué es un dato y por qué importa su taxonomía?

Un **dato** es una representación de una característica de una entidad, observación o evento. En un dataframe, cada columna representa una variable y cada fila una observación. La taxonomía ayuda a responder preguntas prácticas:

- ¿Tiene sentido calcular un promedio?
- ¿Existe un orden entre las categorías?
- ¿Qué codificación necesita el modelo?
- ¿Qué valores representan ausencia, desconocido o una categoría real?

### Dos perspectivas que conviene separar

| Perspectiva | Pregunta | Ejemplo |
|---|---|---|
| **Tipo almacenado** | ¿Cómo está guardado en Python? | `int64`, `float64`, `object`, `bool` |
| **Significado estadístico** | ¿Qué operaciones representan la realidad? | nominal, ordinal, discreta, continua |

Una columna guardada como número no necesariamente es cuantitativa. Por ejemplo, `1 = rojo`, `2 = azul` sigue siendo **nominal**: los números son etiquetas y no cantidades.

### Escalas y tipos de variable

| Tipo | Característica | Ejemplos | Operaciones razonables |
|---|---|---|---|
| **Binaria** | Dos estados | `yes/no`, `0/1`, presencia/ausencia | proporción, conteo, tasa |
| **Nominal** | Categorías sin orden | trabajo, país, color | frecuencia, moda |
| **Ordinal** | Categorías con orden, sin distancias necesariamente iguales | primaria/secundaria/terciaria, bajo/medio/alto | comparación de orden, mediana |
| **Cuantitativa discreta** | Conteos enteros | número de contactos, hijos, compras | suma, promedio, dispersión |
| **Cuantitativa continua** | Mediciones en una escala | peso, duración, temperatura | operaciones aritméticas y dispersión |

> **Advertencia:** que una variable tenga distribución normal no es un requisito para llamarla cuantitativa. La distribución se estudia después; no define por sí sola el tipo de variable.

Referencias: [pandas `select_dtypes`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html) · [pandas `CategoricalDtype`](https://pandas.pydata.org/docs/user_guide/categorical.html) · [scikit-learn: encoding categorical features](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html)

##  Caso de estudio: campañas de marketing bancario

Trabajaremos con `bank.csv`. Cada fila representa un contacto de una campaña telefónica y `y` indica si la persona contrató el producto ofrecido.

Antes de ejecutar el código, predice la clasificación de estas variables:

- `age`, `balance`, `duration`
- `job`, `marital`, `education`, `month`
- `default`, `housing`, `loan`, `y`

La predicción importa: la clasificación que hagas guiará la exploración y la transformación posterior.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
df_bank = pd.read_csv('Data/bank.csv')

In [ ]:
df_bank.head()

In [ ]:
df_bank.tail()

In [ ]:
df_bank.shape

In [ ]:
df_bank.columns

In [ ]:
df_bank.info()

### Lectura rápida del esquema

`info()` permite revisar tamaño, tipos almacenados y valores no nulos. Esta salida es un diagnóstico inicial, no una clasificación completa: una variable `object` puede contener categorías, texto libre o fechas mal interpretadas.

En la siguiente tabla verificaremos qué tipo de dato observa `pandas` en cada columna.

In [ ]:
schema_bank = pd.DataFrame({
    'tipo_python': df_bank.dtypes.astype(str),
    'valores_unicos': df_bank.nunique(),
    'faltantes': df_bank.isna().sum()
})
schema_bank

## Variables cualitativas y cuantitativas

`select_dtypes()` clasifica por el tipo almacenado, lo que es útil como primer filtro. Después debemos revisar el significado de cada columna y documentar excepciones.

In [ ]:
df_bank.head(2)

In [ ]:
# Obteniendo datos cuantitativos
df_bank_cuantitativos = df_bank.select_dtypes(include='number')
df_bank_cuantitativos.head()

In [ ]:
df_bank_cuantitativos.columns

#### Cuantitativas discretas y continuas

La separación `number` / `object` no basta. En este dataset, `age`, `balance` y `duration` son medidas cuantitativas; `campaign`, `pdays` y `previous` son conteos o códigos numéricos cuyo significado merece una revisión adicional.

Observa sus rangos y valores únicos antes de elegir una transformación:

In [ ]:
columnas_revisar = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
df_bank[columnas_revisar].agg(['min', 'max', 'nunique']).T

In [ ]:
#Obteniendo datos cualitativos
df_bank_categoricos = df_bank.select_dtypes(include='object')
df_bank_categoricos.head()

In [ ]:
df_bank_categoricos.columns

### Variables binarias

Las columnas `default`, `housing`, `loan` y `y` tienen dos categorías. La codificación `yes/no` conserva legibilidad; si un algoritmo necesita números, puede mapearse explícitamente. No conviene asumir que todas las columnas binarias son objetivos: `y` es la variable respuesta en este caso y las otras son predictoras.

In [ ]:
#Otra alternativa para obtener datos categóricos
df_bank.select_dtypes(exclude='number')

In [ ]:
df_bank_categoricos['education'].nunique(), df_bank_categoricos['education'].unique()

In [ ]:
df_bank_categoricos['education'].value_counts()

In [ ]:
#Guardamos las columnas originales de df_bank
columns_bank = df_bank.columns
columns_bank

In [ ]:
#Guardar las columnas de las variables cuantitivas
columns_cuantitativas = df_bank_cuantitativos.columns
columns_cuantitativas

In [ ]:
#Guardar las columnas de las variables categóricas
columns_categoricas = df_bank_categoricos.columns
columns_categoricas

In [ ]:
# Categóricos (Multiestado o cualitativos) --->  (nominales, ordinales)
df_bank_categoricos.head()

In [ ]:
# Valores únicos y categorías de la columna default
df_bank_categoricos['default'].unique(), df_bank_categoricos['default'].nunique()

In [ ]:
# Lista de columnas y sus categorías
for col in columns_categoricas:
    print(f'* La columna {col} tiene {df_bank_categoricos[col].nunique()} categorías y son:\n {df_bank_categoricos[col].unique()}')

### Variables ordinales: el orden debe declararse

`education` representa niveles educativos con un orden razonable. Aun así, la distancia entre niveles no tiene por qué ser igual: pasar de `primary` a `secondary` no equivale necesariamente a pasar de `secondary` a `tertiary`.

`month` tiene una secuencia temporal, pero es una variable cíclica: diciembre y enero están próximos en el calendario. Para un modelo suele ser mejor usar variables seno/coseno o una codificación categórica que imponer una distancia lineal.

In [ ]:
# `education` tiene un orden natural; `month` es temporal y conviene tratarlo aparte.
ordinales_cat = ['education']
nominales_cat = [
    columna for columna in df_bank_categoricos.columns
    if columna not in ordinales_cat and columna != 'y'
]
binarias_cat = ['default', 'housing', 'loan', 'y']


In [ ]:
# dataframes con datos ordinales categóricos
df_bank_categoricos_ord = df_bank_categoricos[ordinales_cat]
df_bank_categoricos_ord.head()

In [ ]:
# Declarar el orden evita que una transformación posterior lo invente o lo pierda.
education_order = ['primary', 'secondary', 'tertiary']
education_dtype = pd.api.types.CategoricalDtype(
    categories=education_order,
    ordered=True
)
df_bank_categoricos_ord = df_bank_categoricos[ordinales_cat].copy()
df_bank_categoricos_ord['education'] = df_bank_categoricos_ord['education'].astype(education_dtype)
df_bank_categoricos_ord['education'].dtype

### Categóricas Nominales

In [ ]:
nominales_cat = [
    columna for columna in df_bank_categoricos.columns
    if columna not in ordinales_cat and columna not in binarias_cat
]
nominales_cat

In [ ]:
nominales_cat

In [ ]:
# Base de datos con datos nominales categóricos
df_bank_categoricos_nom = df_bank_categoricos[nominales_cat]
df_bank_categoricos_nom

## Valores faltantes: identificar antes de imputar

Un valor faltante no siempre significa lo mismo: puede ser una medición no realizada, una respuesta omitida o un valor que no aplica. Antes de decidir, cuantifica el problema y consulta el significado de la variable.

En este cuaderno solo hacemos una introducción. La comparación detallada de métodos de imputación continúa en el módulo de **Tratamiento de datos faltantes**.

In [ ]:
#Cargamos un dataset de peliculas
df_movie = pd.read_csv('Data/movie_metadata.csv')
df_movie.head()

In [ ]:
#Obtenemos la información general del dataframe
df_movie.info()

In [ ]:
# missing_summary


### Manejando datos Faltantes (intro)

In [ ]:
# Opción 1: eliminar filas completas solo cuando la pérdida sea aceptable.


In [ ]:
#verificar que no se tengan valores faltantes


## tratar datos faltantes en columnas numéricas

In [ ]:
# Para columnas numéricas


In [ ]:
#promedio


In [ ]:
# Opción 2: imputar una columna numérica con la mediana.
# La mediana suele ser más resistente a valores extremos que el promedio.


## Estrategias iniciales para valores faltantes

La estrategia depende del tipo de variable y del contexto. Elimina filas solo si la pérdida es pequeña y no introduce sesgo; imputa con estadísticas calculadas en el conjunto de entrenamiento cuando prepares un modelo.

Para categorías, `Unknown` debe distinguirse de una categoría real. En un proyecto, documenta cuántos valores fueron imputados y por qué.

In [ ]:
df_movie['color'].unique()

## Actividad de clase

Responde y justifica tus decisiones. No existe una única respuesta correcta si explicas el criterio.

1. Clasifica `job`, `education`, `month`, `campaign` y `y` según su significado estadístico.
2. ¿Por qué no sería correcto calcular el promedio de `job` aunque se codifique con números?
3. ¿Qué problema puede aparecer si se codifica `month` como `1, 2, ..., 12` y se usa esa columna directamente en un modelo?
4. Calcula la proporción de personas con `y == 'yes'` y compárala por nivel de `education`.
5. Elige entre eliminar o imputar los faltantes de `movie_metadata.csv`. Reporta cuántas filas o valores afecta tu decisión.

### Reto

Construye un diccionario llamado `taxonomia` con tres llaves: `binarias`, `nominales` y `ordinales`. Después verifica que ninguna columna categórica quede sin clasificar.